In [15]:
import wandb
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
wandb.login()
api = wandb.Api()

wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: diana-onutu (oellm-team). Use `wandb login --relogin` to force relogin


In [33]:
runs = api.runs("oellm-team/gidd-gidd")
for run in runs:
    print(run.name, run.id)

small-mdlm-fineweb-4gpu-1node-snls hhicijrc
small-mdlm-fineweb-1gpu-1node-snls 0wmmboco
small-gidd-fineweb-pu=0.0-8gpu-2node-snls gx4jv5yf
small-gidd-fineweb-pu=0.0-4gpu-1node-snls 9q5m45zj
small-gidd-fineweb-pu=0.0-16gpu-4node-snls ozgc2g8a
small-gidd-fineweb-pu=0.0-1gpu-1node-snls 90vz1t06
small-gidd-fineweb-pu=0.0-2gpu-1node-snls i9xfn5u7
small-gidd-fineweb-pu=0.0-16gpu-4node-snls toryh3h2
small-gidd-fineweb-pu=0.0-8gpu-2node-snls ae1lk7qd
small-gidd-fineweb-pu=0.0-1gpu-1node-lrdn 20ljce1n
small-gidd-fineweb-pu=0.0-16gpu-4node-lrdn gah5du24
small-gidd-fineweb-pu=0.0-32gpu-8node-snls-h100 5t600f5m
small-gidd-fineweb-pu=0.0-4gpu-1node-snls ay5m0yiz
tiny-gidd-fineweb-pu=0.0-1gpu-1node-snls-h100 u8i15qxg
small-gidd-fineweb-pu=0.0-4gpu-1node-lrdn gc38r3ha
tiny-gidd-fineweb-pu=0.0-1gpu-1node-snls-h100 q8d30g7a
small-gidd-fineweb-pu=0.0-xgpu-xnode-snls-xbs vuzu6lsc
small-gidd-fineweb-pu=0.0-xgpu-xnode-snls-xbs gokfu9dp
small-gidd-fineweb-pu=0.0-16gpu-4node-snls-64bs sh29jiah
small-gidd-fin

In [38]:
runs = api.runs("oellm-team/gidd-gidd")
for run in runs:
    if "1B" in run.name and "64gpu" in run.name:
        print(run.name, run.id)

1B-gidd-fineweb-pu=0.0-64gpu-16node-lrdn-a100-8mbs d6tk0gp8
1B-gidd-fineweb-pu=0.0-64gpu-16node-snls-a100-8mbs tt63u77w
1B-gidd-fineweb-pu=0.0-64gpu-16node-snls-h100-8mbs jmhnpnao
1B-gidd-fineweb-pu=0.0-64gpu-16node-snls-a100-8mbs ix31y022
1B-gidd-fineweb-pu=0.0-64gpu-16node-snls-a100-8mbs 1xn8evn4
1B-gidd-fineweb-pu=0.0-64gpu-16node-snls-a100-8mbs 1kcwbgcv
1B-gidd-fineweb-pu=0.0-64gpu-16node-snls-a100-8mbs ufx3llwn
1B-gidd-fineweb-pu=0.0-64gpu-16node-snls-h100-8mbs q6ukez1v
1B-gidd-fineweb-pu=0.0-64gpu-16node-snls-h100-8mbs o4kgpttz
1B-gidd-fineweb-pu=0.0-64gpu-16node-snls-h100-8mbs e5w5vol6
1B-gidd-fineweb-pu=0.0-64gpu-16node-snls-h100-8mbs 30fg328j
1B-gidd-fineweb-pu0.0-64gpu-16node-jsc-a100-8bs 8kg1z7gm


In [6]:
# Replace with your project and run info
entity = "oellm-team"
project = "gidd-gidd"
run_id = "b3flyymv"

run = api.run(f"{entity}/{project}/{run_id}")

In [7]:
# This will give you a list of dicts with metrics per logged step
history = run.history(keys=["train/tokens_per_sec"])  # optional: filter only columns you need

In [8]:
history

,_step,train/tokens_per_sec
0,9,164128.174471
1,19,176927.094618
2,29,174139.667962
3,39,167127.680886
4,49,172235.687477
...,...,...
195,1959,182785.968862
196,1969,160160.026566
197,1979,165547.330340
198,1989,148336.835512


In [9]:
history[history['_step'] > 1000]["train/tokens_per_sec"].mean()

172152.19696780865

In [34]:
entity = "oellm-team"
projects = ["gidd", "ar"]
hpcs = ["snls-a100", "snls-h100", "lrdn", "jsc"]
gpu_counts = [1, 4, 16, 32, 64]

# Initialize stats
stats = {prj: {hpc: {gpu: [] for gpu in gpu_counts} for hpc in hpcs} for prj in projects}
print(stats)

{'gidd': {'snls-a100': {1: [], 4: [], 16: [], 32: [], 64: []}, 'snls-h100': {1: [], 4: [], 16: [], 32: [], 64: []}, 'lrdn': {1: [], 4: [], 16: [], 32: [], 64: []}, 'jsc': {1: [], 4: [], 16: [], 32: [], 64: []}}, 'ar': {'snls-a100': {1: [], 4: [], 16: [], 32: [], 64: []}, 'snls-h100': {1: [], 4: [], 16: [], 32: [], 64: []}, 'lrdn': {1: [], 4: [], 16: [], 32: [], 64: []}, 'jsc': {1: [], 4: [], 16: [], 32: [], 64: []}}}


In [39]:
entity = "oellm-team"
projects = ["gidd", "ar"]
hpcs = ["snls-a100", "snls-h100", "lrdn", "jsc"]
gpu_counts = [1, 4, 16, 32, 64]

# Initialize stats
stats = {prj: {hpc: {gpu: [] for gpu in gpu_counts} for hpc in hpcs} for prj in projects}

for prj in projects:
    project_name = f"gidd-{prj}"
    runs = api.runs(f"{entity}/{project_name}")

    for run in runs:
        if "1B" not in run.name:
            continue

        for proj_key in projects:
            if proj_key not in run.name:
                continue

            for hpc in hpcs:
                if hpc not in run.name:
                    continue

                # Determine GPU count
                for gpu in gpu_counts:
                    
                    if f"{gpu}gpu" in run.name:
                        print(run.name, run.id)
                        history = run.history(keys=["train/tokens_per_sec"])
                        throughput = history[history['_step'] > 1000]["train/tokens_per_sec"].mean()
                        stats[proj_key][hpc][gpu].append(throughput)
                        break  # stop checking further GPU counts once matched

1B-gidd-fineweb-pu=0.0-1gpu-1node-lrdn-a100-8mbs
1B-gidd-fineweb-pu=0.0-1gpu-1node-lrdn-a100-8mbs u8fwjeqm
1B-gidd-fineweb-pu=0.0-4gpu-1node-lrdn-a100-8mbs
1B-gidd-fineweb-pu=0.0-4gpu-1node-lrdn-a100-8mbs
1B-gidd-fineweb-pu=0.0-4gpu-1node-lrdn-a100-8mbs a42kkpbd
1B-gidd-fineweb-pu=0.0-8gpu-2node-lrdn-a100-8mbs
1B-gidd-fineweb-pu=0.0-8gpu-2node-lrdn-a100-8mbs
1B-gidd-fineweb-pu=0.0-8gpu-2node-lrdn-a100-8mbs
1B-gidd-fineweb-pu=0.0-8gpu-2node-lrdn-a100-8mbs
1B-gidd-fineweb-pu=0.0-8gpu-2node-lrdn-a100-8mbs
1B-gidd-fineweb-pu=0.0-1gpu-1node-snls-a100-8mbs
1B-gidd-fineweb-pu=0.0-1gpu-1node-snls-a100-8mbs 4nu60665
1B-gidd-fineweb-pu=0.0-16gpu-4node-lrdn-a100-8mbs
1B-gidd-fineweb-pu=0.0-16gpu-4node-lrdn-a100-8mbs
1B-gidd-fineweb-pu=0.0-16gpu-4node-lrdn-a100-8mbs
1B-gidd-fineweb-pu=0.0-16gpu-4node-lrdn-a100-8mbs ahf1bog5
1B-gidd-fineweb-pu=0.0-32gpu-8node-lrdn-a100-8mbs
1B-gidd-fineweb-pu=0.0-32gpu-8node-lrdn-a100-8mbs
1B-gidd-fineweb-pu=0.0-32gpu-8node-lrdn-a100-8mbs
1B-gidd-fineweb-pu=0.0-32g

In [36]:
stats

{'gidd': {'snls-a100': {1: [10896.021071920271,
    10965.468663510594,
    10973.859304422369,
    10967.661117348947,
    10833.366007728635],
   4: [41431.69022472788,
    325938.0066077711,
    41602.38216284637,
    41212.906299186645,
    41011.34347171548,
    40435.33770201286,
    326233.9113486024,
    326023.95621294033,
    326146.2262238289,
    41467.22578802804,
    326091.4251553537],
   16: [89721.67178037365,
    89487.71819740153,
    89612.79280201657,
    89658.82428752446,
    89300.84382545072],
   32: [172287.2428465683,
    172346.18987983066,
    172133.8649982669,
    172152.19696780865,
    172010.651720154],
   64: []},
  'snls-h100': {1: [22088.633400277922,
    18667.60605143874,
    21992.299059966845,
    21847.982186957925,
    20697.58115065037],
   4: [81310.44228863948,
    918315.4375047788,
    71822.75645771292,
    78128.3947648908,
    76807.8909767293,
    73782.614575915,
    996314.8955378251,
    998904.4562991874,
    998766.0086442744,
  

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- GIDD ---
axes[0].plot(gpus, eff_lrnd_gidd, marker='o', linestyle='-', label='Leonardo (A100, 64GB)')
axes[0].plot(gpus, eff_snls_a100_gidd, marker='o', linestyle='-', label='Snellius (A100, 40GB)')
axes[0].plot(gpus, eff_snls_h100_gidd, marker='o', linestyle='-', label='Snellius (H100, 94GB)')
axes[0].plot(gpus, eff_jsc_a100_gidd, marker='o', linestyle='-', label='Juwels (A100, 40GB)')
axes[0].axhline(y=1.0, color='gray', linestyle='--', linewidth=1)
axes[0].axhline(y=0.8, color='gray', linestyle='--', linewidth=1)
axes[0].axhline(y=0.7, color='gray', linestyle='--', linewidth=1)
axes[0].axhline(y=0.6, color='gray', linestyle='--', linewidth=1)
axes[0].axhline(y=0.5, color='gray', linestyle='--', linewidth=1)
axes[0].set_title('GIDD Scaling Efficiency')
axes[0].set_xlabel('Number of GPUs')
axes[0].set_ylabel('Efficiency: T(N)/(N * T(1))')
axes[0].set_xticks(gpus)
axes[0].set_ylim(0, 1.1)
axes[0].legend()

# --- AR ---
axes[1].plot(gpus, eff_lrnd_ar, marker='o', linestyle='-', label='Leonardo (A100, 64GB)')
axes[1].plot(gpus, eff_snls_a100_ar, marker='o', linestyle='-', label='Snellius (A100, 40GB)')
axes[1].plot(gpus, eff_snls_h100_ar, marker='o', linestyle='-', label='Snellius (H100, 94GB)')
axes[1].plot(gpus, eff_jsc_a100_ar, marker='o', linestyle='-', label='Juwels (A100, 40GB)')
axes[1].axhline(y=1.0, color='gray', linestyle='--', linewidth=1)
axes[1].axhline(y=0.8, color='gray', linestyle='--', linewidth=1)
axes[1].axhline(y=0.7, color='gray', linestyle='--', linewidth=1)
axes[1].axhline(y=0.6, color='gray', linestyle='--', linewidth=1)
axes[1].axhline(y=0.5, color='gray', linestyle='--', linewidth=1)
axes[1].set_title('AR Scaling Efficiency')
axes[1].set_xlabel('Number of GPUs')
axes[1].set_xticks(gpus)
axes[1].legend()

plt.suptitle('Scaling efficiency 1.5B model (1.2 non-emb.) per GPUs\ncontext_length=512, mbs=8, DDP')
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()